# Permuting and sampling data to calculate the null hypothesis

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
data_loc = '../data/server_ready'

files = [os.path.join(data_loc, f) for f in os.listdir(data_loc) if (not f.startswith('._')) and f.endswith('.csv') and ('null-' not in f)]
print(files)

['../data/server_ready/cha_corpus.csv', '../data/server_ready/cha_corpus-2.csv', '../data/server_ready/cha_corpus-3.csv', '../data/server_ready/grace_corpus.csv', '../data/server_ready/cha_corpus-4.csv', '../data/server_ready/cha_corpus-coraal.csv']


In [3]:
null_output_file = 'null-corpora/{}-null-corpus.parquet'
null_output_file = os.path.join(data_loc, null_output_file)
null_output_file

'../data/server_ready/null-corpora/{}-null-corpus.parquet'

## Setting up the null-test data

### Load, permute, save

In [4]:
print_summaries = True

In [6]:
for f in tqdm(files):
    print(f)
    df = pd.read_csv(f)

    # print(df[['x_utterance', 'y_utterance']].isna().sum(),'\n')
    df = df.loc[
        (~df['x_utterance'].isna())
        & (~df['y_utterance'].isna())
        # & (~df['x_utterance'].isin(['', ' ']))
        # & (~df['y_utterance'].isin(['', ' ']))
    ]

    if print_summaries:
        print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    print('permuting sentences')
    df['null_y_utterance'] = df['y_utterance'].sample(frac=1).values

    if print_summaries:
        bad = df['null_y_utterance'] == df['y_utterance']
        print(bad.sum(), bad.mean())
        del bad

    df['y_utterance'] = df['null_y_utterance'].values
    del df['null_y_utterance']

    if print_summaries:
        print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    df['x_turn_id'] = df['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)

    df['sample_delta'] = df.groupby(by=['x_turn_id', 'self']).cumcount() + 1

    print('finding examples')
    # a lot of work just to find useful examples ...
    good_examples = df[['corpus', 'x_turn_id', 'self', 'sample_delta']].groupby(by=['corpus', 'x_turn_id', 'self', 'sample_delta']).agg('max').reset_index()
    good_examples = pd.merge(
        left=good_examples.loc[good_examples['self']], left_on='x_turn_id',
        right=good_examples[['x_turn_id', 'self', 'sample_delta']].loc[~good_examples['self']], right_on='x_turn_id',
        how='left'
    )
    good_examples = good_examples.loc[
        ((good_examples['sample_delta_x'] >= 5)
        & (good_examples['sample_delta_x'] <= 70))
        & ((good_examples['sample_delta_y'] >= 5)
        & (good_examples['sample_delta_y'] <= 70))
    ]

    df = df.loc[df['x_turn_id'].isin(good_examples['x_turn_id'])]
    # print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    df['conversation_id'] = df['conversation_id'].astype(str)

    print('saving data')
    for c in tqdm(df['corpus'].unique()):

        sub = df['x_turn_id'].loc[df['corpus'].isin([c])].unique()

        k = int(np.ceil(len(sub)*.1))

        if k > 0:
            sub = np.random.choice(sub, size=(k,), replace=False)
            df_ = df.loc[df['x_turn_id'].isin(sub)]

            print(c, k, df_.shape,)
            if print_summaries:
                print(df_[['x_utterance', 'y_utterance']].isna().sum(), '\n')

            null_output_file_ = str(null_output_file).format(c)
            # save subset to null output file.
            # if os.path.exists(null_output_file):
            #     df_.to_csv(null_output_file, index=False, header=False, encoding='utf-8', mode='a', sep='\t')
            # else:
            df_.to_parquet(null_output_file_)

    print('===][===')

  0%|          | 0/6 [00:00<?, ?it/s]

../data/server_ready/cha_corpus.csv
x_utterance    0
y_utterance    0
dtype: int64 

permuting sentences
19590 0.004212646843213628
x_utterance    0
y_utterance    0
dtype: int64 

finding examples
saving data


  0%|          | 0/7 [00:00<?, ?it/s]

callfriend-eng_n 1776 (106124, 25)
x_utterance    0
y_utterance    0
dtype: int64 

callfriend-eng_s 509 (29981, 25)
x_utterance    0
y_utterance    0
dtype: int64 

callhome 5134 (294079, 25)
x_utterance    0
y_utterance    0
dtype: int64 

instruction-DISPEL 356 (19681, 25)
x_utterance    0
y_utterance    0
dtype: int64 

instruction-Frederiksen 10 (357, 25)
x_utterance    0
y_utterance    0
dtype: int64 

instruction-Graesser 53 (2854, 25)
x_utterance    0
y_utterance    0
dtype: int64 

instruction-med_school 20 (816, 25)
x_utterance    0
y_utterance    0
dtype: int64 

===][===
../data/server_ready/cha_corpus-2.csv
x_utterance    0
y_utterance    0
dtype: int64 

permuting sentences
5547 0.0016421605705612447
x_utterance    0
y_utterance    0
dtype: int64 

finding examples
saving data


  0%|          | 0/3 [00:00<?, ?it/s]

...

grace 2209 (121819, 16)
x_utterance    0
y_utterance    0
dtype: int64 

===][===
../data/server_ready/cha_corpus-4.csv
x_utterance    0
y_utterance    0
dtype: int64 

permuting sentences
24230 0.0029324831754375304
x_utterance    0
y_utterance    0
dtype: int64 

finding examples
saving data


  0%|          | 0/15 [00:00<?, ?it/s]

MICASE-advising 409 (25019, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-colloquia 773 (43923, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-defense 334 (19829, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-discussion 603 (33938, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-interviews 177 (10603, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-labs 1000 (60141, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-lel 1002 (50171, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-les 2078 (114848, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-meetings 664 (39198, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-office_hours 1977 (117864, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-seminars 1146 (67960, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-service_encounters 457 (26716, 20)
x_utterance    0
y_utterance    0
dtype: int64 

MICASE-student_pr

  0%|          | 0/16 [00:00<?, ?it/s]

se0-ag1 1504 (109973, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se0-ag2 1374 (139350, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se0-ag3 1283 (132820, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se0-ag4 585 (54116, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se1-ag1 1429 (95333, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se1-ag2 874 (52651, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se1-ag3 913 (64592, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se1-ag4 681 (40302, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se2-ag1 1096 (78326, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se2-ag2 645 (39236, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se2-ag3 1038 (67409, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se2-ag4 1192 (73598, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se3-ag1 1205 (90841, 19)
x_utterance    0
y_utterance    0
dtype: int64 

se3-ag2 501 (32941, 19)
x_utterance    0

### Stitch files into a single composite whole

In [4]:
corpora_are_in = os.path.join(data_loc, 'null-corpora')
corpora_are_in

'../data/server_ready/null-corpora'

In [5]:
corpora = [f for f in os.listdir(corpora_are_in) if (not f.startswith('._')) and f.endswith('.parquet')]
len(corpora)

102

In [6]:
df = []

In [7]:
for f in corpora:
    # df += [pd.read_table(os.path.join(corpora_are_in,f), sep='\t')]
    df += [pd.read_parquet(os.path.join(corpora_are_in,f))]
    # print(f, df[-1].shape)
    # print(df[-1][['x_utterance', 'y_utterance']].isna().sum())
    # print('===][===\n')
df = pd.concat(df, ignore_index=True)

In [8]:
df[['x_utterance', 'y_utterance']].isna().mean()

x_utterance    0.0
y_utterance    0.0
dtype: float64

In [9]:
# df['x_turn_id'].loc[df['y_utterance'].isna()].nunique()

In [10]:
null_output_file = os.path.join(data_loc,'null-corpus.parquet')

In [11]:
df.to_parquet(null_output_file)

#### Test that stitched doc is okay

In [12]:
null_output_file = os.path.join(data_loc,'null-corpus.parquet')

In [13]:
df = pd.read_parquet(null_output_file)
# df = pd.read_csv(null_output_file)
# df = pd.read_table(null_output_file, sep='\t')
df.shape

(4110515, 32)

In [14]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64